# Deploy Nemotron Speech Studio on Brev

Run all cells to deploy the complete colocated stack:

1. Nemotron ASR Streaming NIM on GPU 0
2. Riva Translate 4B Instruct v2 NIM on GPU 1
3. The published Nemotron Speech Studio UI and transcript server

The browser connects only to port `5101`. The application reaches both NIMs by container name on a private Docker network.

## Deploy the Translate NIM

1. Open [Riva Translate 4B Instruct v2](https://build.nvidia.com/nvidia/riva-translate-4b-instruct-v2) and accept its terms.
2. Deploy this Launchable with an `NGC_API_KEY` that can pull the NIM.
3. Open this notebook in the deployed instance and select **Run All**.
4. Wait for the `Riva Translate v2 is ready.` message. The notebook pulls `nvcr.io/nim/nvidia/riva-translate-4b-instruct-v2:2.0.8`, starts it on GPU 1, and verifies a translation request.

No NVCF function ID is required. Studio calls the self-hosted NIM at `http://riva-translate:8000/v1/chat/completions` over the private Docker network.


## Brev Launchable settings

- **Instance:** 2 × A100 80 GB GPUs
- **Storage:** 256 GiB or more
- **Required secret parameter:** `NGC_API_KEY`
- **Network:** authenticated Brev Secure Link for HTTP port `5101`; no public TCP/UDP rule
- **Source:** this repository
- **Notebook:** `scripts/deploy_nemotron_speech_studio_launchable.ipynb`

Accept the NGC terms for the two NIM containers before deployment. Do not expose ports `50051`, `9000`, or `8000` in Brev; they are loopback-only for diagnostics and use a private Docker network for application traffic.


In [ ]:
import getpass
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request

ASR_IMAGE = "nvcr.io/nim/nvidia/nemotron-asr-streaming:1.3.0"
TRANSLATE_IMAGE = "nvcr.io/nim/nvidia/riva-translate-4b-instruct-v2:2.0.8"
APP_IMAGE = os.environ.get(
    "NEMOTRON_SPEECH_STUDIO_IMAGE",
    "nvcr.io/0965475784688077/riva/nemotron-speech-studio:pr-19-a138691",
)
ASR_CONTAINER = "nemotron-asr"
TRANSLATE_CONTAINER = "riva-translate"
APP_CONTAINER = "nemotron-speech-studio"
DOCKER_NETWORK = "nemotron-speech-studio"
ASR_PROFILE = "name=nemotron-asr-streaming,type=en-US,batch_size=32"


def run(command, *, env=None, capture_output=False):
    print("+", " ".join(command))
    return subprocess.run(
        command,
        env=env,
        text=True,
        check=True,
        capture_output=capture_output,
    )


def remove_container(name):
    inspected = subprocess.run(
        ["docker", "container", "inspect", name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if inspected.returncode == 0:
        run(["docker", "rm", "--force", name])


def http_status(url, timeout=5):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return response.status


def http_json(url, *, payload=None, timeout=30):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        method="GET" if body is None else "POST",
        headers={"Content-Type": "application/json", "Accept": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.status, json.loads(response.read().decode("utf-8"))


def wait_for_http(name, container, url, timeout_seconds=1800):
    deadline = time.monotonic() + timeout_seconds
    print(f"Waiting for {name} at {url} ...")
    while time.monotonic() < deadline:
        try:
            if 200 <= http_status(url) < 300:
                print(f"{name} is ready.")
                return
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            pass
        state = subprocess.run(
            ["docker", "inspect", "--format", "{{.State.Running}}", container],
            text=True,
            capture_output=True,
        )
        if state.returncode != 0 or state.stdout.strip() != "true":
            subprocess.run(["docker", "logs", "--tail", "100", container], check=False)
            raise RuntimeError(f"{name} exited before becoming ready")
        time.sleep(10)
    subprocess.run(["docker", "logs", "--tail", "100", container], check=False)
    raise TimeoutError(f"{name} did not become ready within {timeout_seconds} seconds")


In [ ]:
ngc_api_key = os.environ.get("NGC_API_KEY") or getpass.getpass("NGC API key: ")
if not ngc_api_key:
    raise ValueError("NGC_API_KEY is required")

runtime_env = os.environ.copy()
runtime_env["NGC_API_KEY"] = ngc_api_key
print("NGC credentials loaded without displaying the key.")


In [ ]:
if shutil.which("docker") is None:
    raise RuntimeError("Docker is required")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("NVIDIA drivers are required")

run(["docker", "info"], capture_output=True)
gpu_result = run(
    ["nvidia-smi", "--query-gpu=index,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True,
)
gpus = []
for line in gpu_result.stdout.splitlines():
    index, memory_mib = [part.strip() for part in line.split(",")]
    gpus.append((int(index), int(memory_mib)))

if len(gpus) < 2:
    raise RuntimeError(f"Two GPUs are required; found {len(gpus)}")
if any(memory_mib < 75_000 for _, memory_mib in gpus[:2]):
    raise RuntimeError(f"Two 80 GB-class GPUs are required; found {gpus[:2]}")

driver_version = run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True,
).stdout.splitlines()[0].strip()
driver_components = tuple(int(part) for part in driver_version.split("."))
if driver_components < (580, 65, 6):
    raise RuntimeError(
        f"NVIDIA driver 580.65.06 or newer is required for the "
        f"CUDA 13 translation image; found {driver_version}"
    )

docker_root = run(
    ["docker", "info", "--format", "{{.DockerRootDir}}"],
    capture_output=True,
).stdout.strip()
free_gib = shutil.disk_usage(docker_root).free / (1024**3)
if free_gib < 120:
    raise RuntimeError(
        f"At least 120 GiB free Docker storage is required; "
        f"found {free_gib:.1f} GiB in {docker_root}"
    )
print(
    f"Prerequisites passed: {len(gpus)} GPUs, "
    f"{free_gib:.1f} GiB free Docker storage in {docker_root}."
)


In [ ]:
login = subprocess.run(
    ["docker", "login", "nvcr.io", "--username", "$oauthtoken", "--password-stdin"],
    input=ngc_api_key,
    text=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if login.returncode != 0:
    raise RuntimeError("NGC authentication failed; verify NGC_API_KEY and accepted NIM terms")

try:
    for image in (ASR_IMAGE, TRANSLATE_IMAGE, APP_IMAGE):
        run(["docker", "pull", image])
finally:
    subprocess.run(
        ["docker", "logout", "nvcr.io"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )

network_exists = subprocess.run(
    ["docker", "network", "inspect", DOCKER_NETWORK],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0
if not network_exists:
    run(["docker", "network", "create", DOCKER_NETWORK])

for container_name in (APP_CONTAINER, ASR_CONTAINER, TRANSLATE_CONTAINER):
    remove_container(container_name)

for volume_name in ("nemotron-asr-cache", "riva-translate-cache"):
    run(["docker", "volume", "create", volume_name])
    run(
        [
            "docker", "run", "--rm",
            "--user", "0",
            "--entrypoint", "chown",
            "--volume", f"{volume_name}:/cache",
            ASR_IMAGE,
            "-R", "1000:1000", "/cache",
        ]
    )


In [ ]:
run(
    [
        "docker", "run", "--detach",
        "--name", ASR_CONTAINER,
        "--restart", "unless-stopped",
        "--network", DOCKER_NETWORK,
        "--runtime", "nvidia",
        "--gpus", "device=0",
        "--shm-size", "8g",
        "--env", "NGC_API_KEY",
        "--env", "NIM_HTTP_API_PORT=9000",
        "--env", "NIM_GRPC_API_PORT=50051",
        "--env", f"NIM_TAGS_SELECTOR={ASR_PROFILE}",
        "--publish", "127.0.0.1:50051:50051",
        "--publish", "127.0.0.1:9000:9000",
        "--volume", "nemotron-asr-cache:/opt/nim/.cache",
        ASR_IMAGE,
    ],
    env=runtime_env,
)

run(
    [
        "docker", "run", "--detach",
        "--name", TRANSLATE_CONTAINER,
        "--restart", "unless-stopped",
        "--network", DOCKER_NETWORK,
        "--runtime", "nvidia",
        "--gpus", "device=1",
        "--shm-size", "16g",
        "--env", "NGC_API_KEY",
        "--publish", "127.0.0.1:8000:8000",
        "--volume", "riva-translate-cache:/opt/nim/.cache",
        TRANSLATE_IMAGE,
    ],
    env=runtime_env,
)

wait_for_http(
    "Nemotron ASR Streaming",
    ASR_CONTAINER,
    "http://127.0.0.1:9000/v1/realtime/health",
)
wait_for_http(
    "Riva Translate v2",
    TRANSLATE_CONTAINER,
    "http://127.0.0.1:8000/v1/health/ready",
)


In [ ]:
run(
    [
        "docker", "run", "--detach",
        "--name", APP_CONTAINER,
        "--restart", "unless-stopped",
        "--network", DOCKER_NETWORK,
        "--publish", "127.0.0.1:5101:5101",
        "--env", "PORT=5101",
        "--env", "RIVA_SERVER=nemotron-asr:9000",
        "--env", "RIVA_PROTOCOL=http",
        "--env", "TRANSLATION_BACKEND=hosted",
        "--env", "TRANSLATION_HOSTED_CHAT_URL=http://riva-translate:8000/v1/chat/completions",
        "--env", "TRANSLATION_MODEL=nvidia/riva-translate-4b-instruct-v2",
        APP_IMAGE,
    ]
)
wait_for_http(
    "Nemotron Speech Studio",
    APP_CONTAINER,
    "http://127.0.0.1:5101/v1/api/health",
    timeout_seconds=180,
)


In [ ]:
health_status, health = http_json("http://127.0.0.1:5101/v1/api/health")
if health_status != 200:
    raise RuntimeError(f"Application health check returned HTTP {health_status}")
if health.get("servers", {}).get("riva", {}).get("status") != "online":
    raise RuntimeError(f"Application cannot reach ASR: {health}")
if not health.get("servers", {}).get("translation", {}).get("configured"):
    raise RuntimeError(f"Application did not configure local translation: {health}")

translation_status, translation = http_json(
    "http://127.0.0.1:8000/v1/chat/completions",
    payload={
        "model": "nvidia/riva-translate-4b-instruct-v2",
        "messages": [
            {"role": "system", "content": "en-es"},
            {"role": "user", "content": "Hello, this is a translation test."},
        ],
        "temperature": 0,
        "max_tokens": 128,
    },
    timeout=180,
)
if translation_status != 200 or not translation.get("choices"):
    raise RuntimeError(f"Translation smoke test failed: {translation}")

print("Deployment verified end to end.")
print("UI + transcript server: http://<BREV_INSTANCE_ADDRESS>:5101")
print("ASR and translation ports remain private to the instance.")
run(
    [
        "docker", "ps",
        "--filter", f"name={APP_CONTAINER}",
        "--filter", f"name={ASR_CONTAINER}",
        "--filter", f"name={TRANSLATE_CONTAINER}",
        "--format", "table {{.Names}}\t{{.Status}}\t{{.Image}}",
    ]
)


## Access

Preferred: select **Open Nemotron Speech Studio** on the deployed Launchable. Brev serves port `5101` through an authenticated HTTPS Secure Link.

The UI, `/v1/api` HTTP routes, and `/v1/api` WebSocket upgrades use the same Secure Link origin. The browser sends the Brev authentication cookies automatically. The Studio backend reaches `nemotron-asr:9000` and `riva-translate:8000` over the private Docker network, so neither NIM needs a tunnel or exposed port.

For local access through the Brev CLI instead:

```bash
brev refresh
brev port-forward <instance-name> --port 5101:5101
```

Then open `http://127.0.0.1:5101`. If local port `5101` is occupied, use `--port 15101:5101` and open `http://127.0.0.1:15101`. Run `brev refresh` again after an instance restart if its address changes.

## Operations

The three containers use `restart unless-stopped`. Re-running the notebook replaces them while preserving the ASR and translation model-cache volumes.

```bash
docker logs -f nemotron-speech-studio
docker logs -f nemotron-asr
docker logs -f riva-translate
docker rm -f nemotron-speech-studio nemotron-asr riva-translate
```
